# YOLOv8n + DINOv2 v2: Spatial Patch-Token Fusion on Pascal VOC

This notebook trains **only** the improved v2 fusion model. It uses DINOv2 **spatial patch tokens** (location-specific features) instead of the single global class token, projects to 64 channels, and adds a learnable gate.

**Previous run results (already completed):**

| Model | Precision | Recall | mAP50 | mAP50-95 | Inference ms/img |
|---|---:|---:|---:|---:|---:|
| YOLOv8n baseline | 0.6768 | 0.6427 | 0.6852 | 0.4768 | 2.68 |
| YOLOv8n + DINOv2 v1 (global token) | 0.4617 | 0.3911 | 0.3691 | 0.2182 | 36.88 |

This notebook produces the **v2** row using spatial patch-token fusion.

In [ ]:
# 1. Setup Environment and Dependencies
import os
import platform
import random
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.142"], check=True)

import numpy as np
import torch
import torchvision

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Platform:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else platform.platform())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=False)
else:
    raise RuntimeError("Enable a GPU accelerator before training.")

In [ ]:
# 2. Project Configuration
RUNTIME_ROOT = Path("/kaggle/working" if IS_KAGGLE else "/content" if IS_COLAB else Path.cwd())
WORK_DIR = RUNTIME_ROOT / "yolov8_dinov2_voc_v2"
SOURCE_DIR = WORK_DIR / "fusion_source"
ARTIFACT_DIR = RUNTIME_ROOT / "yolov8_dinov2_artifacts_v2"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_YAML = WORK_DIR / "yolov8-dino-v2.yaml"

DATA = "VOC.yaml"
IMAGE_SIZE = 640
BATCH_SIZE = 2
EPOCHS = 10
WORKERS = 2
DEVICE = 0

print("Work directory:", WORK_DIR)
print("Artifacts:", ARTIFACT_DIR)
print("Train settings:", {"epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE, "device": DEVICE})

In [ ]:
# 3. Define the v2 Fusion Module (Spatial Patch Tokens + Gated Fusion)
import importlib.util
import sys
import contextlib
import os
import warnings
from torch import nn
from torchvision import transforms as T

MODULE_SOURCE = r'''from __future__ import annotations

import contextlib
import os
import warnings
from pathlib import Path

import torch
from torch import nn
from torchvision import transforms as T


class ConvDummy(nn.Module):
    """Pass the original image through the YOLO graph unchanged."""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x


class DINOv2(nn.Module):
    """Frozen DINOv2 with spatial patch-token features and gated projection.

    Improvements over v1:
    1. Uses patch tokens (spatially varying) instead of the global class token.
    2. Projects 384 DINO channels to 64 output channels (not 3).
    3. Adds a learnable sigmoid gate that starts near zero so the pretrained
       YOLO detector is initially unaffected.
    """
    def __init__(self, output_channels=64):
        super().__init__()
        with warnings.catch_warnings(), open(os.devnull, "w", encoding="utf-8") as sink:
            warnings.simplefilter("ignore")
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        self.model.eval()
        self.model.requires_grad_(False)

        self.projector = nn.Conv2d(384, output_channels, kernel_size=1)
        nn.init.normal_(self.projector.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.projector.bias)

        self.gate = nn.Parameter(torch.zeros(1, output_channels, 1, 1))

        self.transform = T.Compose([
            T.Resize((518, 518), antialias=True),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def train(self, mode=True):
        super().train(mode)
        self.model.eval()
        return self

    def forward(self, input_tensor):
        with torch.no_grad():
            processed = self.transform(input_tensor)
            patch_tokens = self.model.get_intermediate_layers(processed, n=1, reshape=True)[0]
        height = max(1, input_tensor.shape[2] // 32)
        width = max(1, input_tensor.shape[3] // 32)
        projected = self.projector(patch_tokens)
        gated = torch.sigmoid(self.gate) * projected * 0.1
        return torch.nn.functional.interpolate(
            gated, size=(height, width), mode="bilinear", align_corners=False,
        )


# Cache for the ORIGINAL parse_model and its source so register_modules is
# idempotent — safe to call multiple times (cell 3 imports + cell 5 builds).
_ORIG_PARSE_MODEL = None
_ORIG_PARSE_SRC = None

def register_modules():
    """Register custom modules and patch parse_model for correct channel tracking.

    Ultralytics' parse_model uses ``else: c2 = ch[f]`` for unknown modules,
    which sets the *output* channel count to the *input* channel count (3).
    For DINOv2 (384->64), this is wrong: it reports 3 instead of 64,
    breaking every downstream layer.  We patch parse_model so DINOv2
    uses ``c2 = args[0]`` (the output_channels parameter) WITHOUT stripping
    args (unlike the TorchVision branch which does ``args = [*args[1:]]``).

    This function is idempotent: it caches the original parse_model source
    on first call and reuses it on subsequent calls, so calling
    inspect.getsource on an already-patched function won't fail.
    """
    global _ORIG_PARSE_MODEL, _ORIG_PARSE_SRC
    import inspect
    import textwrap
    from ultralytics.nn import tasks

    tasks.ConvDummy = ConvDummy
    tasks.DINOv2 = DINOv2

    # Only capture original on the FIRST call (before we patch it).
    # On subsequent calls, reuse the cached original source.
    if _ORIG_PARSE_SRC is None:
        _ORIG_PARSE_MODEL = tasks.parse_model
        _ORIG_PARSE_SRC = inspect.getsource(_ORIG_PARSE_MODEL)

    # Insert a DINOv2-specific branch right before the TorchVision branch.
    old_branch = "elif m in frozenset({TorchVision, Index}):"
    new_branch = (
        "elif m is DINOv2:\n"
        "            c2 = args[0]\n"
        "            c1 = ch[f]\n"
        "        elif m in frozenset({TorchVision, Index}):"
    )

    assert old_branch in _ORIG_PARSE_SRC, (
        "Could not find TorchVision branch in parse_model source to patch"
    )
    patched_src = _ORIG_PARSE_SRC.replace(old_branch, new_branch)
    patched_src = textwrap.dedent(patched_src)

    # Compile in a namespace that has DINOv2 and all tasks module globals
    namespace = dict(vars(tasks))
    namespace["DINOv2"] = DINOv2
    exec(patched_src, namespace)
    tasks.parse_model = namespace["parse_model"]
'''

MODULE_FILE = SOURCE_DIR / "dino_modules_v2.py"
MODULE_FILE.write_text(MODULE_SOURCE, encoding="utf-8")
spec = importlib.util.spec_from_file_location("dino_modules_v2", MODULE_FILE)
dino_modules_v2 = importlib.util.module_from_spec(spec)
sys.modules["dino_modules_v2"] = dino_modules_v2
spec.loader.exec_module(dino_modules_v2)
dino_modules_v2.register_modules()
print("v2 fusion modules registered (patch tokens, 64 channels, gated).")
print("parse_model patched: DINOv2 now reports c2 = args[0] (64 channels).")

In [ ]:
# 4. Write the v2 Model YAML (64-channel DINO, 20 VOC classes)
MODEL_YAML.parent.mkdir(parents=True, exist_ok=True)
MODEL_YAML.write_text("""\
# YOLOv8n + DINOv2 spatial patch-token fusion (v2).
nc: 20
scale: n
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, ConvDummy, []]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [0, 1, DINOv2, [64]]
head:
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24], 1, Detect, [nc]]
""", encoding="utf-8")
print("v2 model YAML written:", MODEL_YAML)

In [ ]:
# 5. Build the v2 Fused Model and Transfer Pretrained YOLOv8n Weights
from ultralytics import YOLO
from ultralytics.cfg import get_cfg

def build_fused_v2_model():
    dino_modules_v2.register_modules()
    fused = YOLO(str(MODEL_YAML), verbose=False)
    pretrained = YOLO("yolov8n.pt", verbose=False).model.state_dict()
    fused_state = fused.model.state_dict()
    transferred = 0

    for key, value in fused_state.items():
        parts = key.split(".", 2)
        if len(parts) != 3 or parts[0] != "model" or not parts[1].isdigit():
            continue
        layer_index = int(parts[1])
        if 1 <= layer_index <= 10:
            source_index = layer_index - 1
        elif 13 <= layer_index <= 25:
            source_index = layer_index - 3
        else:
            continue
        source_key = f"model.{source_index}.{parts[2]}"
        source_value = pretrained.get(source_key)
        if source_value is None:
            continue
        if source_value.shape == value.shape:
            fused_state[key] = source_value
            transferred += 1
        elif (
            value.ndim == 4
            and source_value.ndim == 4
            and value.shape[0] == source_value.shape[0]
            and value.shape[1] == source_value.shape[1] + 64
            and value.shape[2:] == source_value.shape[2:]
        ):
            padded = torch.zeros_like(value)
            padded[:, :source_value.shape[1]] = source_value
            torch.nn.init.normal_(padded[:, source_value.shape[1]:], mean=0.0, std=0.01)
            fused_state[key] = padded
            transferred += 1

    fused.model.load_state_dict(fused_state)
    print(f"Transferred {transferred} compatible YOLOv8n tensors into v2 fusion model.")

    def freeze_dino_encoder(trainer):
        encoder = trainer.model.model[11].model
        encoder.requires_grad_(False)
        encoder.eval()

    def report_adapter_norm(trainer):
        projector = trainer.model.model[11].projector
        gate = trainer.model.model[11].gate
        print(
            f"v2 adapter epoch {trainer.epoch + 1}: "
            f"proj_norm={float(projector.weight.detach().norm()):.6g}, "
            f"gate_mean={float(torch.sigmoid(gate).mean().detach()):.6g}"
        )

    fused.add_callback("on_train_start", freeze_dino_encoder)
    fused.add_callback("on_train_epoch_end", report_adapter_norm)
    return fused

print("v2 fusion model builder ready.")

In [ ]:
# 6. One-Batch Shape and Gradient Check
fused_check = build_fused_v2_model()
fused_check.model.to(f"cuda:{DEVICE}").eval()

context_seen = {}
hook = fused_check.model.model[11].register_forward_hook(
    lambda module, inputs, output: context_seen.update(shape=tuple(output.shape))
)
with torch.inference_mode():
    prediction = fused_check.model(torch.zeros(1, 3, 320, 320, device=f"cuda:{DEVICE}"))
hook.remove()

assert context_seen["shape"][0] == 1 and context_seen["shape"][1] == 64, context_seen
assert torch.isfinite(prediction[0]).all()
print("v2 context shape:", context_seen["shape"])
print("Prediction shape:", tuple(prediction[0].shape))

training_model = fused_check.model.train()
training_model.args = get_cfg()
training_model.criterion = training_model.init_criterion()
synthetic_batch = {
    "img": torch.rand(1, 3, 320, 320, device=f"cuda:{DEVICE}"),
    "batch_idx": torch.tensor([0.0], device=f"cuda:{DEVICE}"),
    "cls": torch.tensor([[0.0]], device=f"cuda:{DEVICE}"),
    "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.2]], device=f"cuda:{DEVICE}"),
}
projector = training_model.model[11].projector
initial = projector.weight.detach().clone()
loss_parts, detached = training_model.loss(synthetic_batch)
loss_parts.sum().backward()
assert projector.weight.grad is not None and torch.isfinite(projector.weight.grad).all()
assert projector.weight.grad.norm() > 0, "v2 projector received no gradient"
optimizer = torch.optim.AdamW([p for p in training_model.parameters() if p.requires_grad], lr=1e-4)
optimizer.step()
delta = (projector.weight.detach() - initial).norm()
assert delta > 0, "Optimizer did not update v2 projector"
print("v2 gradient norm:", float(projector.weight.grad.norm()))
print("v2 projector update:", float(delta))
del fused_check, training_model, optimizer

## Train the v2 Fusion Model

This cell trains only the v2 spatial patch-token fusion model on Pascal VOC. It uses the same settings as the baseline and v1 runs: 10 epochs, 640px, batch 2, AdamW, lr 0.001, FP32, seed 42. The project name is `voc_yolov8n_dinov2_v2_fusion` so it does not overwrite previous runs.

In [ ]:
# 7. Train the v2 Fusion Model on Pascal VOC
fused_v2_model = build_fused_v2_model()
fused_v2_result = fused_v2_model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    amp=False,
    optimizer="AdamW",
    lr0=0.001,
    close_mosaic=2,
    seed=SEED,
    deterministic=True,
    cls_remap=True,
    pretrained=False,
    plots=True,
    project=str(ARTIFACT_DIR / "training"),
    name="voc_yolov8n_dinov2_v2_fusion",
)

V2_WEIGHTS = Path(fused_v2_result.save_dir) / "weights" / "best.pt"
assert V2_WEIGHTS.exists(), V2_WEIGHTS
print("v2 fusion checkpoint:", V2_WEIGHTS)

In [ ]:
# 8. Evaluate the v2 Fusion Model
import csv
import json
import shutil

v2_eval_model = YOLO(str(V2_WEIGHTS), verbose=False)
v2_metrics = v2_eval_model.val(
    data=DATA, imgsz=IMAGE_SIZE, device=DEVICE, plots=True,
    project=str(ARTIFACT_DIR / "evaluation"), name="voc_fusion_v2",
)

v2_result = {
    "model": "YOLOv8n + DINOv2 v2 (patch tokens, 64ch, gated)",
    "precision": float(v2_metrics.box.mp),
    "recall": float(v2_metrics.box.mr),
    "map50": float(v2_metrics.box.map50),
    "map50_95": float(v2_metrics.box.map),
    "inference_ms_per_image": float(v2_metrics.speed.get("inference", float("nan"))),
}
print("v2 evaluation:", v2_result)

In [ ]:
# 9. Three-Way Comparison Table
comparison = [
    {
        "model": "YOLOv8n baseline",
        "precision": 0.6768,
        "recall": 0.6427,
        "map50": 0.6852,
        "map50_95": 0.4768,
        "inference_ms_per_image": 2.68,
    },
    {
        "model": "YOLOv8n + DINOv2 v1 (global token, 3ch)",
        "precision": 0.4617,
        "recall": 0.3911,
        "map50": 0.3691,
        "map50_95": 0.2182,
        "inference_ms_per_image": 36.88,
    },
    {
        "model": "YOLOv8n + DINOv2 v2 (patch tokens, 64ch, gated)",
        "precision": v2_result["precision"],
        "recall": v2_result["recall"],
        "map50": v2_result["map50"],
        "map50_95": v2_result["map50_95"],
        "inference_ms_per_image": v2_result["inference_ms_per_image"],
    },
]

results_json = ARTIFACT_DIR / "voc_comparison_v2.json"
results_csv = ARTIFACT_DIR / "voc_comparison_v2.csv"
results_json.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
with results_csv.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)

print("\nThree-Way Comparison")
print("Model | P | R | mAP50 | mAP50-95 | inference ms/img")
for row in comparison:
    print(
        f"{row['model']} | {row['precision']:.4f} | {row['recall']:.4f} | "
        f"{row['map50']:.4f} | {row['map50_95']:.4f} | "
        f"{row['inference_ms_per_image']:.2f}"
    )
print("Saved:", results_json, results_csv)